# Servidor de inferencia del peaje inteligente (Google Colab)
Alternativa a RunPod con el **mismo código** (`inferencia/app`). Expone la API con un túnel
**Cloudflare** (sin cuenta), igual que el notebook `week_2_servidor_pose_colab_v2.ipynb` del curso.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*.

La URL `https://xxxx.trycloudflare.com` cambia en cada ejecución: cópiala a `INFERENCE_URL` en la web
(archivo `.env` o la pantalla **Estado**, que permite cambiarla sin tocar el código).

## 1. Configuración

In [ ]:
import os, secrets
PUERTO = 8000
REPO_URL = "https://github.com/saraquenta/laboratorio3_peajeIA.git"   # tu repositorio público
DIR_PROYECTO = "/content/peaje"
DIR_INFERENCIA = f"{DIR_PROYECTO}/inferencia"
API_TOKEN = secrets.token_urlsafe(24)       # o fija uno: API_TOKEN = "mi_token"
USAR_DRIVE = False                          # True para leer pesos propios desde Google Drive
DIR_PESOS_DRIVE = "/content/drive/MyDrive/peaje/pesos"
print("Token:", API_TOKEN)

## 2. Código, dependencias y túnel

In [ ]:
if not os.path.isdir(DIR_PROYECTO):
    !git clone --depth 1 {REPO_URL} {DIR_PROYECTO}
!pip install -q -r {DIR_INFERENCIA}/requirements.txt
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared && cloudflared --version
if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    !cp -v {DIR_PESOS_DRIVE}/*.pt {DIR_INFERENCIA}/pesos/ || true
!cd {DIR_INFERENCIA} && python diagnostico.py

## 3. Iniciar servidor y túnel

In [ ]:
import subprocess, threading, time, re, json, urllib.request

entorno = dict(os.environ, API_TOKEN=API_TOKEN, PORT=str(PUERTO))
log = open("/content/inferencia.log", "w")
servidor = subprocess.Popen(["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", str(PUERTO)],
                            cwd=DIR_INFERENCIA, env=entorno, stdout=log, stderr=subprocess.STDOUT)

for _ in range(120):
    try:
        salud = json.loads(urllib.request.urlopen(f"http://localhost:{PUERTO}/health", timeout=5).read())
        if salud["estado"] in ("ok", "error"):
            break
    except Exception:
        salud = {"estado": "arrancando"}
    time.sleep(3)
print("Servidor:", salud)

URL_PUBLICA = None
tunel = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://localhost:{PUERTO}"],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
log_tunel = []
threading.Thread(target=lambda: [log_tunel.append(l) for l in tunel.stdout], daemon=True).start()
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", "".join(log_tunel))
    if m:
        URL_PUBLICA = m.group(0); break
    time.sleep(0.5)
time.sleep(3)
print("\nINFERENCE_URL   =", URL_PUBLICA)
print("INFERENCE_TOKEN =", API_TOKEN)

## 4. Prueba local y a través del túnel

In [ ]:
import numpy as np, cv2, matplotlib.pyplot as plt
datos = urllib.request.urlopen("https://ultralytics.com/images/bus.jpg", timeout=30).read()
cab = {"Content-Type": "image/jpeg", "X-API-Token": API_TOKEN}
res = json.loads(urllib.request.urlopen(urllib.request.Request(f"http://localhost:{PUERTO}/predict", data=datos, headers=cab), timeout=120).read())
print(json.dumps({k: res[k] for k in ("placa_normalizada", "confianza_ocr", "categoria", "ejes", "pasajeros", "latencia_ms")}, indent=2))
vis = urllib.request.urlopen(urllib.request.Request(f"http://localhost:{PUERTO}/predict/visual", data=datos, headers=cab), timeout=120).read()
plt.figure(figsize=(10, 7)); plt.imshow(cv2.imdecode(np.frombuffer(vis, np.uint8), 1)[:, :, ::-1]); plt.axis("off"); plt.show()
try:
    print("Túnel OK:", json.loads(urllib.request.urlopen(f"{URL_PUBLICA}/health", timeout=30).read())["estado"])
except Exception as e:
    print("El túnel aún no responde (espera unos segundos y repite):", e)

## 5. Mantener vivo / detener
Colab corta la sesión tras un rato de inactividad. Deja esta pestaña abierta durante la demo.

In [ ]:
DETENER = False
if DETENER:
    servidor.terminate(); tunel.terminate(); print("Detenidos")
else:
    !tail -n 20 /content/inferencia.log